In [ ]:
# merge all dc8 nasa lfight tracks into one netcdf 

import glob, pandas as pd, monetio as mio, os 



In [ ]:
DC8_DIR = "/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/DC8"

OUT = "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/dc8_data/asiaaq_dc8_merge_all.nc"

RENAME = {
    # lat lon p coords
    "Latitude_BENNETT":          "latitude",
    "Longitude_BENNETT":         "longitude",
    "Static_Pressure_BENNETT":   "pressure_obs",   # hPa 
    "GPS_Altitude_m_DIGANGI":    "altitude",        # m MSL 

    # met 
    "T_GATEBE":  "temperature",       # K  
    "U_GATEBE":  "u",                 # m/s  E-W
    "V_GATEBE":  "v",                 # m/s  N-S
    "W_GATEBE":  "w",                 # m/s  
    "TAS_GATEBE":   "true_air_speed", # m/s  
    "TEDR_GATEBE":  "turb_dissip_log10",  # log10 kW/kg (obs-only)
    "REYN_GATEBE":  "reynolds_log10",     # log10 /m   (obs-only)
    "H2O_DLH_DISKIN": "h2o",          # ppmv water vapor

    # chem
    "O3_ppbv":            "O3",       # ppbv
    "NO_pptv":            "NO",       # pptv
    "NO2_pptv":           "NO2",      # pptv
    "CO_DACOM_DISKIN":    "CO",       # ppbv
    "CH4_DACOM_DISKIN":   "CH4",      # ppbv
    "CO2_7000_ppm_DISKIN":"CO2",      # ppm
    
}

NA = [-999999, -99999, -9999, -8888, -7777]

def read_merge(path, keep):
    with open(path) as fh:
        lines = fh.read().splitlines()
    nlhead = int(lines[0].split(",")[0])
    y, m, d = (int(x) for x in lines[6].split(",")[:3])
    base = pd.Timestamp(year=y, month=m, day=d)

    raw = [c.strip() for c in lines[nlhead - 1].split(",")]
    # de-duplicate repeated column names so read_csv accepts them
    seen, cols = {}, []
    for n in raw:
        if n in seen:
            seen[n] += 1
            cols.append(f"{n}.{seen[n]}")
        else:
            seen[n] = 0
            cols.append(n)

    use = [c for c in keep if c in cols]
    df = pd.read_csv(path, skiprows=nlhead, names=cols,
                     usecols=["Time_Start"] + use, na_values=NA)
    df["time"] = base + pd.to_timedelta(
        pd.to_numeric(df["Time_Start"], errors="coerce"), unit="s")
    return df.drop(columns=["Time_Start"])
    

In [ ]:
keep = list(RENAME)
files = sorted(glob.glob(os.path.join(DC8_DIR, "asiaaq-mrg10_dc8_*.ict")))
df = (pd.concat([read_merge(f, keep) for f in files], ignore_index=True)
        .rename(columns=RENAME).sort_values("time"))

df["pressure_obs"] = df["pressure_obs"] * 100.0      # conv units 
df = df.dropna(subset=["time"]).drop_duplicates(subset="time").sort_values("time")
ds = df.set_index("time").to_xarray()
ds.to_netcdf(OUT)
print(ds)
